# Colab runner — 05 storage analysis
CPU sufficient; reuse an existing tokenized dataset or generate/tokenize only. Push aggregate metrics, not shard files or LMDB data.

In [ ]:
from pathlib import Path
import os,subprocess
repo=Path('/content/pragmatiq-scalable-explainable')
if not repo.exists():
    subprocess.run(['git','clone','-b','research','https://github.com/haider4445/pragmatiq-scalable-explainable.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run(['git','switch','research'],check=True)
subprocess.run(['git','pull','--ff-only','origin','research'],check=True)

In [ ]:
%pip install -e '.[train]'

In [ ]:
from pragmatiq import api
roots=[Path('runs/research_decomposed'),Path('runs/research_baseline'),Path('runs/research_tokenizer')]
root=next((p for p in roots if (p/'tok/shard_manifest.json').exists() and (p/'tok/user_index.lmdb').exists()),None)
if root is None:
    root=Path('runs/research_storage')
    api.synthesize({'n_users':1000,'seed':42},out=root/'raw',n_workers=0,write_report=False)
    api.tokenize(root/'raw',root/'tok',config={'target_vocab':28000,'n_buckets':64})
print(root)

In [ ]:
import sys
subprocess.run([sys.executable,'research/experiments/05_storage_analysis.py','--root',str(root)],check=True)

In [ ]:
import json,pandas as pd
r=json.loads(Path('research/results/05_storage_metrics.json').read_text())
display(pd.DataFrame(r['bands']))
display(pd.DataFrame(r['shards']))
print(r['sample'])

## Push reviewed aggregate metrics
Requires Colab secret GITHUB_TOKEN and GitHub CLI. Never commit customer-level data.

In [ ]:
from google.colab import userdata
import shutil
assert shutil.which('gh'), 'Install GitHub CLI first: !apt-get install -y gh'
env=dict(os.environ,GH_TOKEN=userdata.get('GITHUB_TOKEN'))
subprocess.run(['gh','auth','setup-git'],env=env,check=True)
subprocess.run(['git','config','user.name','haider4445'],check=True)
if subprocess.run(['git','config','user.email'],capture_output=True).returncode!=0:
    subprocess.run(['git','config','user.email','34925624+haider4445@users.noreply.github.com'],check=True)
f='research/results/05_storage_metrics.json'
subprocess.run(['git','add','--',f],check=True)
subprocess.run(['git','diff','--cached','--stat'],check=True)

In [ ]:
subprocess.run(['git','commit','-m','experiment: record shard and LMDB storage metrics','--',f],check=True)
subprocess.run(['git','pull','--rebase','origin','research'],env=env,check=True)
subprocess.run(['git','push','origin','research'],env=env,check=True)
del env